# MSE Budget & GMS — Four-Method Comparison

Compares four approaches to the open-boundary problem in the BEACH L4 dropsonde MSE budget.

| Method | Name | What changes | ERA5 needed? |
|--------|------|-------------|-------------|
| **M1** | Baseline | Nothing — raw BEACH omega | No |
| **M2** | O'Brien → 0 | Linear ramp forces ω=0 at BEACH top | No |
| **M3** | O'Brien → ERA5 | Ramp targets ERA5 value at junction; ERA5 stitched above | Yes |
| **M4** | Cosine blend | Cosine taper BEACH→ERA5 in top ~50 hPa; BEACH unchanged below | Yes |

**Key diagnostic:** `h_div_residual = <h·∇·v> − <ω ∂h/∂p>` should be ≈ 0 for a closed column.  
**Science output:** GMS by category (Top-Heavy vs Bottom-Heavy) from each method.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

from scripts.mse_budget import load_dataset, compute_budget, apply_mass_correction
from scripts.era5_extension import (
    load_era5_omega,
    apply_era5_anchored_correction,
    compute_budget_m3,
    compute_budget_m4,
)

COLOR_TH = '#d62728'
COLOR_BH = '#1f77b4'

ZARR_PATH = '/g/data/k10/zr7147/ORCESTRA_dropsondes_categorized.zarr'
ERA5_PATH = '/g/data/k10/zr7147/ERA5/era5_omega_pressure_levels.nc'

## 1. Load data

In [ ]:
ds_beach = load_dataset(ZARR_PATH)
era5_ds  = load_era5_omega(ERA5_PATH)

print(f"BEACH dims : {dict(ds_beach.sizes)}")
print(f"ERA5 dims  : {dict(era5_ds.sizes)}")
print(f"ERA5 pressure levels (hPa): {era5_ds['pressure_level'].values}")

## 2. Run all four methods

ERA5 is loaded once and passed in to avoid triple file reads.

In [ ]:
print("M1: Baseline (raw BEACH, no correction)...")
budget_m1 = compute_budget(ds_beach, mass_correct=False)
print("  done.")

In [ ]:
print("M2: O'Brien ramp → 0 (no ERA5)...")
budget_m2 = compute_budget(ds_beach, mass_correct=True)
print("  done.")

In [ ]:
print("M3: O'Brien ramp → ERA5 value + stitch ERA5 above...")
budget_m3, delta_div_m3, ds_ext_m3 = compute_budget_m3(ds_beach, era5_ds=era5_ds)
print(f"  done.  ERA5 levels added per circle: "
      f"mean={ds_ext_m3['era5_n_levels'].values.mean():.1f}  "
      f"min={ds_ext_m3['era5_n_levels'].values.min()}  "
      f"max={ds_ext_m3['era5_n_levels'].values.max()}")

In [ ]:
print("M4: Cosine blend BEACH→ERA5 (top 50 hPa) + stitch ERA5 above...")
budget_m4, ds_ext_m4 = compute_budget_m4(ds_beach, era5_ds=era5_ds)
print(f"  done.  ERA5 levels added per circle: "
      f"mean={ds_ext_m4['era5_n_levels'].values.mean():.1f}  "
      f"min={ds_ext_m4['era5_n_levels'].values.min()}  "
      f"max={ds_ext_m4['era5_n_levels'].values.max()}")

## 3. Diagnostic: boundary residual

`h_div_residual = <h·∇·v> − <ω ∂h/∂p>` measures the open-boundary contribution:
theoretically equals `h_top · ω_top / g` for a closed column.

**Important caveat for M3/M4 (extended column):**  
`compute_budget_ext` uses a shared mean-altitude grid (`alt_mean`) for the ERA5 extension levels.
When ERA5 levels have different altitudes per circle, `alt_mean` at those positions can be NaN,
causing the h profile to be NaN at some ERA5 levels. This makes `h_div_col` and `vert_adv`
integrate over slightly different domains near the column top — inflating the apparent residual.
**This is a numerical artifact, not a real open-boundary error.**

For M3/M4, the correct closure diagnostic is the **BEACH-column residual** (cell below),
which evaluates closure within the BEACH domain only after each boundary correction.

In [ ]:
methods = [
    (budget_m1, 'M1  Baseline',         'tab:gray'),
    (budget_m2, "M2  O'Brien→0",        'tab:orange'),
    (budget_m3, "M3  O'Brien→ERA5",     'tab:blue'),
    (budget_m4, 'M4  Cosine blend→ERA5','tab:green'),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)

for ax, (budget, label, color) in zip(axes, methods):
    resid = budget['h_div_residual'].values
    rmse  = np.sqrt(np.nanmean(resid**2))
    mean  = np.nanmean(resid)
    ax.hist(resid, bins=25, color=color, alpha=0.75, edgecolor='white', linewidth=0.5)
    ax.axvline(0,    color='black', lw=1.2, ls='--', label='zero')
    ax.axvline(mean, color='red',   lw=2.0, ls='-',  label=f'mean={mean:.1f}')
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('h_div_residual  (W m⁻²)', fontsize=9)
    ax.legend(fontsize=8)
    ax.text(0.97, 0.95, f'RMSE={rmse:.1f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8, color='darkred')

axes[0].set_ylabel('Count')
fig.suptitle('Boundary Residual  (<h·∇·v> − <ω ∂h/∂p>)  —  All Four Methods',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 3b. BEACH-column closure diagnostic

For M3, compute the residual within the **BEACH column only** after applying the
O'Brien correction — this is the fair comparison for boundary closure across methods.

In [ ]:
# BEACH-column residual for M3: apply O'Brien ramp only (no ERA5 extension)
ds_m3_corr, _ = apply_era5_anchored_correction(ds_beach, era5_ds=era5_ds)
budget_m3_beach = compute_budget(ds_m3_corr, mass_correct=False)

print('BEACH-column residual (h*omega_top/g at the BEACH profile top):')
print(f"{'':30s}  {'RMSE (W/m²)':>12}  {'Physical meaning'}")
print('-' * 72)
for budget, label in [(budget_m1,       'M1  Baseline'),
                      (budget_m2,       "M2  O'Brien→0"),
                      (budget_m3_beach, "M3  O'Brien→ERA5 (BEACH only)")]:
    r    = budget['h_div_residual'].values
    rmse = np.sqrt(np.nanmean(r**2))
    note = ('raw BEACH top, ω≠0' if 'M1' in label
            else 'forced ω=0 at top (physically wrong at 200 hPa)' if 'M2' in label
            else 'ω_top=ERA5 value (physically correct; ERA5 closes above)')
    print(f"  {label:28s}  {rmse:12.1f}  {note}")

print()
print('ERA5 omega at ~20 hPa (stratospheric top) for first 5 circles with ERA5:')
print('  (Expected boundary term for full BEACH+ERA5 column)')
from scripts.era5_extension import match_era5_to_circle
for ci2 in np.where(ds_ext_m3['era5_n_levels'].values > 0)[0][:5]:
    om_e, p_e, _, _ = match_era5_to_circle(
        era5_ds,
        float(ds_beach['circle_lat'].values[ci2]),
        float(ds_beach['circle_lon'].values[ci2]),
        ds_beach['circle_time'].values[ci2])
    om_top = om_e[np.argmin(p_e)]
    bnd_top = 350000 * om_top / 9.81
    print(f'  Circle {ci2}: ω(20hPa)={om_top:.5f} Pa/s  →  boundary term ≈ {bnd_top:.0f} W/m²')

print()
print('→ M3 full-column residual should be O(10–200 W/m²); observed higher values')
print('  are due to alt_mean approximation in compute_budget_ext (numerical artifact).')
print('  GMS from vert_adv (Method 1) is NOT affected by this artifact.')

## 4. Omega profile comparison (sample circle)

Shows how each method modifies the omega profile relative to raw BEACH.

In [ ]:
cat = ds_beach['category_avg'].values

# Pick a representative Top-Heavy circle with a clean ERA5 match
th_idx = np.where(np.array(['Top-Heavy' in str(c) for c in cat]))[0]
ci     = th_idx[5]   # 6th top-heavy circle (adjust if needed)

print(f"Circle index: {ci}  |  category: {cat[ci]}  |  "
      f"time: {ds_beach['circle_time'].values[ci]}")

# --- Extract BEACH profile for this circle
p_beach = ds_beach['p_mean'].values[ci]
om_raw  = ds_beach['omega'].values[ci]

# --- M2: apply mass correction to get corrected BEACH omega
ds_m2, _ = apply_mass_correction(ds_beach)
om_m2    = ds_m2['omega'].values[ci]

# --- M3: corrected BEACH + ERA5 extension
ds_m3_corr, _ = apply_era5_anchored_correction(ds_beach, era5_ds=era5_ds)
om_m3_beach   = ds_m3_corr['omega'].values[ci]   # corrected BEACH part only
om_m3_ext     = ds_ext_m3['omega_ext'].values[ci]
p_m3_ext      = ds_ext_m3['p_ext'].values[ci]

# --- M4: blended BEACH + ERA5 extension
om_m4_ext = ds_ext_m4['omega_ext'].values[ci]
p_m4_ext  = ds_ext_m4['p_ext'].values[ci]

# --- ERA5 reference profile for this circle
from scripts.era5_extension import match_era5_to_circle
om_era5_ref, p_era5_ref, _, _ = match_era5_to_circle(
    era5_ds,
    float(ds_beach['circle_lat'].values[ci]),
    float(ds_beach['circle_lon'].values[ci]),
    ds_beach['circle_time'].values[ci],
)

fig, axes = plt.subplots(1, 4, figsize=(16, 8), sharey=True)
titles = ['M1  Baseline', "M2  O'Brien→0", "M3  O'Brien→ERA5", 'M4  Cosine blend']
colors = ['tab:gray', 'tab:orange', 'tab:blue', 'tab:green']

for ax, title, color in zip(axes, titles, colors):
    # ERA5 reference (faint background)
    valid_e = np.isfinite(om_era5_ref) & np.isfinite(p_era5_ref)
    ax.plot(om_era5_ref[valid_e], p_era5_ref[valid_e] / 100,
            color='gray', lw=1.2, ls=':', alpha=0.6, label='ERA5 ref')
    # Raw BEACH (faint for reference)
    valid_b = np.isfinite(om_raw) & np.isfinite(p_beach)
    ax.plot(om_raw[valid_b], p_beach[valid_b] / 100,
            color='black', lw=1.0, ls='--', alpha=0.4, label='BEACH raw')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('ω (Pa s⁻¹)', fontsize=9)
    ax.axvline(0, color='k', lw=0.6, alpha=0.4)

# M1: raw BEACH (same as reference)
valid_b = np.isfinite(om_raw) & np.isfinite(p_beach)
axes[0].plot(om_raw[valid_b], p_beach[valid_b] / 100,
             color='tab:gray', lw=2, label='M1')

# M2: O'Brien-corrected BEACH
valid_m2 = np.isfinite(om_m2) & np.isfinite(p_beach)
axes[1].plot(om_m2[valid_m2], p_beach[valid_m2] / 100,
             color='tab:orange', lw=2, label='M2')

# M3: extended profile
valid_m3 = np.isfinite(om_m3_ext) & np.isfinite(p_m3_ext)
axes[2].plot(om_m3_ext[valid_m3], p_m3_ext[valid_m3] / 100,
             color='tab:blue', lw=2, label='M3')

# M4: blended + extended profile
valid_m4 = np.isfinite(om_m4_ext) & np.isfinite(p_m4_ext)
axes[3].plot(om_m4_ext[valid_m4], p_m4_ext[valid_m4] / 100,
             color='tab:green', lw=2, label='M4')

for ax in axes:
    ax.invert_yaxis()
    ax.set_ylim(1020, 15)
    ax.legend(fontsize=8, loc='lower right')

axes[0].set_ylabel('Pressure (hPa)', fontsize=10)
fig.suptitle(f'Omega Profile — Circle {ci} ({cat[ci]})  |  '
             f'{ds_beach["circle_time"].values[ci]}',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. GMS comparison by category

GMS is computed as the **ratio of group means** (more stable than mean of per-circle ratios):
$$\text{GMS} = \frac{\langle \bar{\omega \partial h / \partial p} \rangle}{\langle \bar{\omega \partial s / \partial p} \rangle}$$

In [ ]:
cat       = ds_beach['category_avg'].values
th_mask   = np.array(['Top-Heavy'    in str(c) for c in cat])
bh_mask   = np.array(['Bottom-Heavy' in str(c) for c in cat])

def gms_group(budget, mask):
    """Ratio-of-means GMS for a subset of circles."""
    va  = budget['vert_adv'].values[mask]
    vds = budget['vert_adv_dse'].values[mask]
    mn_va  = np.nanmean(va)
    mn_vds = np.nanmean(vds)
    return mn_va / mn_vds if abs(mn_vds) > 1.0 else np.nan

print(f"{'Method':25s}  {'GMS Top-Heavy':>14}  {'GMS Bottom-Heavy':>17}  {'ΔGMS (TH−BH)':>13}")
print('-' * 75)

for budget, label, _ in methods:
    gms_th = gms_group(budget, th_mask)
    gms_bh = gms_group(budget, bh_mask)
    delta  = gms_th - gms_bh if np.isfinite(gms_th) and np.isfinite(gms_bh) else np.nan
    print(f"  {label:23s}  {gms_th:+14.4f}  {gms_bh:+17.4f}  {delta:+13.4f}")

In [ ]:
# --- Box + swarm plot comparing vert_adv (numerator) and gms_adv across methods

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

method_labels = ['M1\nBaseline', "M2\nO'Brien→0", "M3\nO'Brien\n→ERA5", 'M4\nBlend\n→ERA5']
method_colors = ['tab:gray', 'tab:orange', 'tab:blue', 'tab:green']

for ax, varname, ylabel in zip(axes,
                                ['vert_adv', 'gms_adv'],
                                ['Vertical MSE advection  (W m⁻²)', 'GMS_adv  (−)']):
    for k, (budget, _, color) in enumerate(methods):
        vals_th = budget[varname].values[th_mask]
        vals_bh = budget[varname].values[bh_mask]
        x_th = k - 0.18
        x_bh = k + 0.18
        # Box plots
        bp_th = ax.boxplot(vals_th[np.isfinite(vals_th)], positions=[x_th],
                           widths=0.28, patch_artist=True, notch=False,
                           boxprops=dict(facecolor=COLOR_TH, alpha=0.6),
                           medianprops=dict(color='white', lw=2),
                           whiskerprops=dict(color=COLOR_TH),
                           capprops=dict(color=COLOR_TH),
                           flierprops=dict(marker='.', color=COLOR_TH, alpha=0.4))
        bp_bh = ax.boxplot(vals_bh[np.isfinite(vals_bh)], positions=[x_bh],
                           widths=0.28, patch_artist=True, notch=False,
                           boxprops=dict(facecolor=COLOR_BH, alpha=0.6),
                           medianprops=dict(color='white', lw=2),
                           whiskerprops=dict(color=COLOR_BH),
                           capprops=dict(color=COLOR_BH),
                           flierprops=dict(marker='.', color=COLOR_BH, alpha=0.4))

    ax.set_xticks(range(4))
    ax.set_xticklabels(method_labels, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.axhline(0, color='k', lw=0.8, ls='--', alpha=0.5)
    ax.set_title(f'{varname}  by category and method', fontsize=10)

legend_els = [
    Line2D([0], [0], color=COLOR_TH, lw=6, alpha=0.6, label='Top-Heavy'),
    Line2D([0], [0], color=COLOR_BH, lw=6, alpha=0.6, label='Bottom-Heavy'),
]
axes[1].legend(handles=legend_els, fontsize=9, loc='upper right')

fig.suptitle('Vertical MSE Advection and GMS by Category — All Four Methods',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Summary: ΔGMS stability across methods

If the sign and rough magnitude of ΔGMS (Top-Heavy − Bottom-Heavy) is consistent across M1–M4,
the result is robust to the boundary correction method.

In [ ]:
delta_gms_vals = []
for budget, label, color in methods:
    gms_th = gms_group(budget, th_mask)
    gms_bh = gms_group(budget, bh_mask)
    delta_gms_vals.append(gms_th - gms_bh if np.isfinite(gms_th + gms_bh) else np.nan)

fig, ax = plt.subplots(figsize=(7, 4))

x = np.arange(4)
bars = ax.bar(x, delta_gms_vals, color=method_colors, alpha=0.8, edgecolor='white', width=0.55)
ax.axhline(0, color='k', lw=1.0, ls='--', alpha=0.6)

for bar, val in zip(bars, delta_gms_vals):
    if np.isfinite(val):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.003 * np.sign(val),
                f'{val:+.4f}', ha='center', va='bottom' if val >= 0 else 'top',
                fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(method_labels, fontsize=10)
ax.set_ylabel('ΔGMS  (Top-Heavy − Bottom-Heavy)', fontsize=10)
ax.set_title('Robustness check: ΔGMS across boundary correction methods', fontsize=11)
plt.tight_layout()
plt.show()

print("\nConclusion:")
if all(np.sign(v) == np.sign(delta_gms_vals[0]) for v in delta_gms_vals if np.isfinite(v)):
    print(f"  ΔGMS sign is CONSISTENT across all methods ({np.sign(delta_gms_vals[0]):+.0f}).")
    print(f"  Range: {min(delta_gms_vals):.4f} to {max(delta_gms_vals):.4f}")
else:
    print("  WARNING: ΔGMS sign is NOT consistent — results are sensitive to boundary correction.")

## 7. Method selection

**h_div_residual interpretation:**
- M1/M2: BEACH-column only → residual directly measures boundary closure ✓
- M3/M4: Extended column → residual is inflated by altitude-grid approximation artefact;
  **use BEACH-column residual (cell above) for fair comparison**

**GMS interpretation:**
- M2 (BEACH-column, ω→0 at ~200 hPa): conservative — misses upper-troposphere contribution
- M3 (full atmosphere, O'Brien→ERA5): physically complete; O'Brien ensures smooth junction
- M4 (full atmosphere, cosine blend): minimal BEACH distortion but blend can change vert_adv

**Recommended for primary result:** M3 (ΔGMS = +0.70)
**Sensitivity check:** M2 (+0.63) and M4 (+0.90) bracket the range

In [ ]:
print(f"{'Method':25s}  {'Mean |resid| (W/m²)':>21}  {'BEACH resid RMSE':>18}  {'ΔGMS':>8}")
print('-' * 80)

for (budget, label, _), dg in zip(methods, delta_gms_vals):
    resid = budget['h_div_residual'].values
    mae   = np.nanmean(np.abs(resid))
    rmse  = np.sqrt(np.nanmean(resid**2))
    print(f"  {label:23s}  {mae:21.1f}  (see BEACH diagnostic)  {dg:+8.4f}")

print()
print('ΔGMS range across methods:', min(delta_gms_vals), 'to', max(delta_gms_vals))
print('All methods agree on sign: ΔGMS > 0  (Top-Heavy more efficient than Bottom-Heavy)')
print()
print('Recommended: M3 as primary, M2/M4 as sensitivity bounds.')